# 09 — Curador EVA

Construye una llave única `municipio + año + período + cultivo`. Excluye taxonomías incompatibles, suma producción y área cosechada y recalcula el rendimiento; nunca promedia rendimientos simples.

In [ ]:
from pathlib import Path
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
REPO_REF = 'feature/SCRUM-15'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, 'https://github.com/cybercolombia/suelosabio.git', str(REPO_DIR)], check=True)
    else:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    PIPELINE_DIR = next((p for p in [Path.cwd(), Path.cwd() / 'ClimatePipeline', Path.cwd().parent / 'ClimatePipeline'] if (p / 'DatasetConfig.py').exists()), None)
if PIPELINE_DIR is None:
    raise FileNotFoundError('No se encontró ClimatePipeline.')
sys.path.insert(0, str(PIPELINE_DIR)) if str(PIPELINE_DIR) not in sys.path else None
from DatasetConfig import cargar_configuracion_datasets
from ClimateProcessingUtils import escribir_parquet_atomico, escribir_json_atomico, escribir_texto_atomico
from CropYieldProcessing import CURATION_VERSION, curate_eva
import pandas as pd
from IPython.display import display
CONFIG = cargar_configuracion_datasets(in_colab=IN_COLAB)


In [ ]:
DATASET_ID = 'uejq-wxrr'
CULTIVOS_OBJETIVO = None  # Ejemplo: ['Papa', 'Maíz']; None conserva todos.
EJECUTAR_CURACION = False
SOBRESCRIBIR_RESULTADOS = False
RAW_ROOT = CONFIG.eva_raw_root / f'fuente={DATASET_ID}'
OUTPUT_DIR = CONFIG.processed_root / 'agricultura_curada' / f'version={CURATION_VERSION}'
archivos = sorted(RAW_ROOT.rglob('part-*.parquet')) if RAW_ROOT.exists() else []
print({'curation_version': CURATION_VERSION, 'cultivos': CULTIVOS_OBJETIVO, 'archivos': len(archivos), 'entrada': str(RAW_ROOT), 'salida': str(OUTPUT_DIR), 'ejecutar': EJECUTAR_CURACION})

In [ ]:
resultado_curacion = None
if not EJECUTAR_CURACION:
    print('Curación desactivada. Revise la auditoría cruda y active EJECUTAR_CURACION.')
else:
    if not archivos:
        raise FileNotFoundError(f'No hay Parquet EVA en {RAW_ROOT}')
    crudo = pd.concat([pd.read_parquet(path) for path in archivos], ignore_index=True)
    resultado_curacion = curate_eva(crudo, crops=CULTIVOS_OBJETIVO)
    tablas = {
        'eva_curada.parquet': resultado_curacion.curated,
        'exclusiones.parquet': resultado_curacion.exclusions,
        'reconciliacion.parquet': resultado_curacion.reconciliation,
        'resumen_cobertura.parquet': resultado_curacion.summary,
    }
    for nombre, tabla in tablas.items():
        escribir_parquet_atomico(tabla, OUTPUT_DIR / nombre, SOBRESCRIBIR_RESULTADOS)
    manifest = {'estado': 'COMPLETA_CON_REVISION_PENDIENTE', 'curation_version': CURATION_VERSION, 'dataset_id': DATASET_ID, 'cultivos_objetivo': CULTIVOS_OBJETIVO, 'filas_curadas': len(resultado_curacion.curated), 'filas_excluidas': len(resultado_curacion.exclusions), 'llave': ['codigo_municipio', 'anio', 'periodo', 'cultivo'], 'columnas_no_predictoras': ['produccion_t', 'area_cosechada_ha']}
    escribir_json_atomico(manifest, OUTPUT_DIR / 'manifest.json', SOBRESCRIBIR_RESULTADOS)
    diccionario = '# Diccionario EVA curada\n\n- `rendimiento_t_ha`: target recalculado como producción / área cosechada.\n- `produccion_t` y `area_cosechada_ha`: auditoría del target; prohibidas como predictores.\n- `metodologia_desde_2022`: marca el quiebre metodológico documentado por UPRA.\n'
    escribir_texto_atomico(diccionario, OUTPUT_DIR / 'data_dictionary.md', SOBRESCRIBIR_RESULTADOS)
    display(resultado_curacion.reconciliation)
